In [1]:
# CELL 1: INSTALL EVERYTHING REQUIRED

!apt-get update -qq
!apt-get install -y -qq stockfish
!pip -q install python-chess ipywidgets

print("Installation complete! Run the next cell.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package stockfish.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../stockfish_16-1build1_amd64.deb ...
Unpacking stockfish (16-1build1) ...
Setting up stockfish (16-1build1) ...
Processing triggers for man-db (2.12.0-4build2) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 14.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 53.3 MB/s eta 0:00:00
Installation complete! Run the next cell.


In [2]:

# ============================================================
# ADVANCED CHESS AI — GOOGLE COLAB
# Human vs AI | AI vs AI | Stockfish | Analysis | Move History
# ============================================================

import os
import shutil
import chess
import chess.engine
import ipywidgets as widgets

from IPython.display import display, clear_output
from IPython.display import HTML
from IPython.display import Javascript

# -------------------- START ENGINE ---------------------------

ENGINE_PATH = shutil.which("stockfish")

if ENGINE_PATH is None:
    for path in ["/usr/games/stockfish", "/usr/bin/stockfish"]:
        if os.path.exists(path):
            ENGINE_PATH = path
            break

if ENGINE_PATH is None:
    raise RuntimeError(
        "Stockfish not found. Run the installation cell first."
    )

# Close an engine left running by a previous execution.
try:
    engine.quit()
except Exception:
    pass

engine = chess.engine.SimpleEngine.popen_uci(ENGINE_PATH)
engine.configure({"Threads": 2, "Hash": 128})

# -------------------- GAME STATE -----------------------------

board = chess.Board()
selected_square = None
flipped = False
last_move = None

# -------------------- VISUAL THEME ---------------------------

PIECES = {
    "K": "♔", "Q": "♕", "R": "♖", "B": "♗", "N": "♘", "P": "♙",
    "k": "♚", "q": "♛", "r": "♜", "b": "♝", "n": "♞", "p": "♟"
}

LIGHT = "#EEEED2"
DARK = "#769656"
SELECTED = "#F6F669"
LAST_MOVE = "#BACB44"
LEGAL = "#A9D18E"

# -------------------- SETTINGS ------------------------------

mode = widgets.Dropdown(
    options=[
        ("Player vs AI", "human"),
        ("AI vs AI", "ai")
    ],
    value="human",
    description="Mode:",
    layout=widgets.Layout(width="240px")
)

human_color = widgets.Dropdown(
    options=[("White", "white"), ("Black", "black")],
    value="white",
    description="Your side:",
    layout=widgets.Layout(width="240px")
)

white_skill = widgets.IntSlider(
    value=15, min=0, max=20, step=1,
    description="White AI:",
    continuous_update=False,
    layout=widgets.Layout(width="340px")
)

black_skill = widgets.IntSlider(
    value=15, min=0, max=20, step=1,
    description="Black AI:",
    continuous_update=False,
    layout=widgets.Layout(width="340px")
)

think_time = widgets.FloatSlider(
    value=0.5, min=0.1, max=3.0, step=0.1,
    description="Think (s):",
    readout_format=".1f",
    continuous_update=False,
    layout=widgets.Layout(width="340px")
)

# -------------------- CHESSBOARD -----------------------------

square_buttons = {}

def create_board_grid():
    global square_buttons

    children = []

    for row in range(8):
        for col in range(8):
            button = widgets.Button(
                description="",
                tooltip="",
                layout=widgets.Layout(
                    width="54px",
                    height="54px",
                    padding="0px",
                    margin="0px"
                ),
                style=widgets.ButtonStyle(
                    font_weight="bold",
                    text_color="#202020"
                )
            )

            button.add_class("chess-square")
            square_buttons[(row, col)] = button
            children.append(button)

    grid = widgets.GridBox(
        children=children,
        layout=widgets.Layout(
            width="432px",
            grid_template_columns=" ".join(["54px"] * 8),
            grid_template_rows=" ".join(["54px"] * 8),
            grid_gap="0px",
            border="3px solid #333333"
        )
    )

    for (row, col), button in square_buttons.items():
        button.on_click(
            lambda btn, r=row, c=col: on_square_click(r, c)
        )

    return grid

board_grid = create_board_grid()

status = widgets.HTML()
evaluation = widgets.HTML()
move_history = widgets.HTML()

# -------------------- BOARD RENDERING ------------------------

def display_coordinates(row, col):
    if flipped:
        return 7 - row, 7 - col
    return 7 - row, col

def refresh_board():
    legal_targets = set()

    if selected_square is not None:
        legal_targets = {
            move.to_square
            for move in board.legal_moves
            if move.from_square == selected_square
        }

    for row in range(8):
        for col in range(8):
            file_idx, rank_idx = display_coordinates(row, col)
            sq = chess.square(file_idx, rank_idx)

            button = square_buttons[(row, col)]
            piece = board.piece_at(sq)

            button.description = (
                PIECES[piece.symbol()] if piece else " "
            )
            button.tooltip = chess.square_name(sq)

            # Standard alternating square colors
            color = LIGHT if (file_idx + rank_idx) % 2 else DARK

            if last_move and sq in (
                last_move.from_square, last_move.to_square
            ):
                color = LAST_MOVE

            if sq in legal_targets:
                color = LEGAL

            if sq == selected_square:
                color = SELECTED

            button.style.button_color = color

            if piece and piece.color == chess.WHITE:
                button.style.text_color = "#FFFFFF"
            else:
                button.style.text_color = "#171717"

    if board.is_checkmate():
        status.value = (
            "<h3 style='color:#C62828'>"
            f"Checkmate! {'Black' if board.turn else 'White'} wins."
            "</h3>"
        )
    elif board.is_stalemate():
        status.value = "<h3>Draw by stalemate.</h3>"
    elif board.is_game_over():
        status.value = (
            f"<h3>Game over: {board.result()} — "
            f"{board.outcome().termination.name.replace('_', ' ').title()}</h3>"
        )
    else:
        turn = "White" if board.turn else "Black"
        message = f"<h3>Turn: {turn}</h3>"

        if board.is_check():
            message += "<b style='color:#C62828'>CHECK!</b>"

        status.value = message

    san_moves = []
    for i in range(0, len(board.move_stack), 2):
        white_move = board.san(board.move_stack[i]) if False else None

    # Use the move stack to build readable algebraic notation.
    temp = chess.Board()
    for i, move in enumerate(board.move_stack):
        san = temp.san(move)
        if i % 2 == 0:
            san_moves.append(f"{i // 2 + 1}. {san}")
        else:
            san_moves[-1] += f" {san}"
        temp.push(move)

    move_history.value = (
        "<div style='max-height:180px;overflow-y:auto;"
        "font-family:monospace;font-size:14px;padding:8px;"
        "background:#f3f4f6;color:#111;border-radius:6px'>"
        + ("<br>".join(san_moves[-30:]) if san_moves else "No moves yet.")
        + "</div>"
    )

# -------------------- AI ENGINE ------------------------------

def set_engine_strength(color):
    skill = white_skill.value if color == chess.WHITE else black_skill.value

    if "Skill Level" in engine.options:
        engine.configure({"Skill Level": skill})

def ai_move():
    global last_move, selected_square

    if board.is_game_over():
        refresh_board()
        return

    set_engine_strength(board.turn)

    result = engine.play(
        board,
        chess.engine.Limit(time=think_time.value)
    )

    last_move = result.move
    selected_square = None
    board.push(result.move)
    refresh_board()

def is_human_turn():
    if mode.value == "ai":
        return False

    human = (
        chess.WHITE if human_color.value == "white"
        else chess.BLACK
    )
    return board.turn == human

# -------------------- PLAYER INTERACTION ---------------------

def on_square_click(row, col):
    global selected_square, last_move

    if board.is_game_over() or not is_human_turn():
        return

    file_idx, rank_idx = display_coordinates(row, col)
    sq = chess.square(file_idx, rank_idx)
    piece = board.piece_at(sq)

    if selected_square is not None:
        move = chess.Move(selected_square, sq)

        moving_piece = board.piece_at(selected_square)

        # Automatically promote to a queen.
        if (
            moving_piece
            and moving_piece.piece_type == chess.PAWN
            and chess.square_rank(sq) in (0, 7)
        ):
            move.promotion = chess.QUEEN

        if move in board.legal_moves:
            last_move = move
            board.push(move)
            selected_square = None
            refresh_board()

            if mode.value == "human" and not board.is_game_over():
                ai_move()

            return

    if piece and piece.color == board.turn:
        selected_square = sq
    else:
        selected_square = None

    refresh_board()

# -------------------- GAME CONTROLS --------------------------

def new_game(_=None):
    global board, selected_square, last_move

    board = chess.Board()
    selected_square = None
    last_move = None

    refresh_board()

    if mode.value == "human" and human_color.value == "black":
        ai_move()

def undo_move(_=None):
    global selected_square, last_move

    selected_square = None

    if not board.move_stack:
        refresh_board()
        return

    # In human mode, undo the AI move and the player's preceding move.
    board.pop()

    if mode.value == "human" and board.move_stack:
        board.pop()

    last_move = board.peek() if board.move_stack else None
    refresh_board()

def flip_board(_=None):
    global flipped
    flipped = not flipped
    refresh_board()

def play_ai_moves(_=None):
    global last_move, selected_square

    if mode.value != "ai":
        status.value = "<b>Switch Mode to AI vs AI first.</b>"
        return

    for _ in range(10):  # Ten half-moves per click
        if board.is_game_over():
            break
        set_engine_strength(board.turn)
        result = engine.play(
            board,
            chess.engine.Limit(time=think_time.value)
        )
        last_move = result.move
        board.push(result.move)
        selected_square = None
        refresh_board()

    refresh_board()

def analyse_position(_=None):
    if board.is_game_over():
        evaluation.value = "<b>Game has ended.</b>"
        return

    info = engine.analyse(
        board,
        chess.engine.Limit(time=max(0.2, think_time.value))
    )

    score = info["score"].white()
    cp = score.score(mate_score=100000)

    if score.is_mate():
        eval_text = f"Mate in {score.mate()} (White's perspective)"
    else:
        eval_text = f"{cp / 100:+.2f} pawns (White's perspective)"

    pv = info.get("pv", [])
    best = board.variation_san(pv[:5]) if pv else "Unavailable"

    evaluation.value = (
        "<div style='padding:10px;background:#e8eef7;"
        "color:#111;border-radius:8px'>"
        f"<b>Evaluation:</b> {eval_text}<br>"
        f"<b>Best line:</b> {best}"
        "</div>"
    )

# -------------------- BUTTONS -------------------------------

new_button = widgets.Button(
    description="New Game", icon="refresh",
    button_style="primary"
)
undo_button = widgets.Button(
    description="Undo", icon="undo"
)
flip_button = widgets.Button(
    description="Flip Board", icon="exchange"
)
ai_button = widgets.Button(
    description="AI vs AI: Run 10 Moves",
    button_style="success", icon="play"
)
analysis_button = widgets.Button(
    description="Analyse Position", icon="search"
)

new_button.on_click(new_game)
undo_button.on_click(undo_move)
flip_button.on_click(flip_board)
ai_button.on_click(play_ai_moves)
analysis_button.on_click(analyse_position)

mode.observe(lambda change: new_game(), names="value")
human_color.observe(
    lambda change: new_game(), names="value"
)

# -------------------- USER INTERFACE -------------------------

title = widgets.HTML("""
<div style="padding:16px;background:#202b38;color:white;
border-radius:10px;margin-bottom:12px">
<h1 style="margin:0">♟ CHESS MASTER AI</h1>
<p style="margin:6px 0 0">Powered by Stockfish • Full legal chess</p>
</div>
""")

settings = widgets.VBox([
    widgets.HTML("<h3>Game Settings</h3>"),
    mode,
    human_color,
    white_skill,
    black_skill,
    think_time,
    widgets.HTML(
        "<small>Skill 0–20; higher skill and longer thinking "
        "usually produce stronger moves. Engine strength also "
        "depends on available compute.</small>"
    )
])

controls = widgets.HBox(
    [new_button, undo_button, flip_button],
    layout=widgets.Layout(flex_flow="row wrap", gap="6px")
)

ai_controls = widgets.HBox(
    [ai_button, analysis_button],
    layout=widgets.Layout(flex_flow="row wrap", gap="6px")
)

game_layout = widgets.VBox([
    title,
    settings,
    widgets.HTML("<h3>Chessboard</h3>"),
    board_grid,
    status,
    controls,
    ai_controls,
    widgets.HTML("<h3>Engine Analysis</h3>"),
    evaluation,
    widgets.HTML("<h3>Move History</h3>"),
    move_history
])

display(game_layout)
refresh_board()

print("Chess Master AI is ready!")
print("Select your mode and click a piece, then its destination.")


Chess Master AI is ready!
Select your mode and click a piece, then its destination.
